Look at the TAC dataset and see how it is structured

In [ ]:
import pandas as pd
import re

# filepaths for the current data we have to work with
tac_path = "../../datasets/tac/baseline/test/test_df.pkl"
conll_test_path = "../../datasets/conll_2003/full/test.txt"
conll_train_path = "../../datasets/conll_2003/full/train.txt"
conll_val_path = "../../datasets/conll_2003/full/val.txt"

# load the TAC data to see where we're supposed to go with CONLL
df = pd.read_pickle(tac_path)
df.head()

[Someone already loaded](https://gist.github.com/morrisalp/006275e076a9cc6e57650a8e18069526) the CONLL2003 dataset into a dataframe, adjust the code so it is more suited to our needs. 

In [ ]:
# throws a file that is in the conll format (tab-delimited) into a dataframe
def read_conll_direct(filename):
    df = pd.read_csv(filename,
                    sep = '\t', header = None, keep_default_na = False,
                    names = ['Label', 'POS', 'CHUNK', 'Word'],
                    quoting = 3, skip_blank_lines = False)
    df['SENTENCE'] = (df.Label == '').cumsum()
    df = df[df.Label != '']
    return df[["Word", "Label"]]

conll_df = read_conll_direct(conll_test_path)
conll_df.head()


Reuse code from `tac_preprocessing.ipynb` to read `.conll` files for CONLL2003 data.

In [ ]:
import os
import re
import pandas as pd
import numpy as np


def dir_to_df(path):
    """
    Takes a filepath to a directory of raw datafiles and creates a Pandas dataframe.

    Function pseudocode / implementation details:
    A) Create a list of files in the raw data directory.
    B) Create a dataframe from the first file.
    C) Add a NaN row for later pre-processing steps.
    D) Loop through remaining files and extend the existing dataframe.
        E) Read in the raw data from the file.
        F) Create a NaN row for later.
        G) Extend the existing dataframe.
    H) Add column headers to the dataframe.
    """

    file_list = [file for file in os.listdir(path) if re.search('.*\.conll', file)]
    data_frames = []

    for filename in file_list:
        file_path = os.path.join(path, filename)
        new_data = pd.read_csv(file_path, sep='\t', encoding="utf-8", header=None,
                               skip_blank_lines=False, names=["Label", "Start", "End", "Word"],
                               escapechar="\\")
        data_frames.append(new_data)
        # Append a NaN row that matches the structure of your data
        nan_row = pd.DataFrame([[np.nan, np.nan, np.nan, np.nan]], columns=["Label", "Start", "End", "Word"])
        data_frames.append(nan_row)

    # Concatenate all data frames (and NaN rows) into a single DataFrame
    data = pd.concat(data_frames, ignore_index=True).reset_index(drop=True)

    return data


def add_sentence_numbers(data):
    """
    Uses the dataframe's NaN rows to assign a sentence number to each token.
    Sentence numbers associate individual tokens with the sentence they belong to.
    """
    sentence_increment, sentence_numbers = 2, ["Sentence: 1"]

    # Assign a sentence number to the first token of each sentence
    for index in range(1, len(data), 1):
        if pd.isna(data.iloc[index]["Label"]):
            sentence_numbers.append("Sentence: " + str(sentence_increment))
            sentence_increment += 1
        else:
            sentence_numbers.append(np.nan)

    data["Sentence #"] = sentence_numbers  # Add a new column for sentence number
    data["Sentence #"] = data["Sentence #"].ffill()  # Forward fill in sentence numbers
    data = data.dropna()  # Drop separator rows (Rows still containing NaN values)
    return data

def get_unique_list_labels(dataframe):
    """
    Extracts a unique list of labels from a DataFrame's 'Label' column after sentences have been grouped.

    Args:
        dataframe (DataFrame): DataFrame with 'Label' column containing lists of labels.

    Returns:
        list: List of unique labels.
    """
    # Use a set comprehension to extract unique labels from each list in the 'Label' column
    unique_labels = {label for label_list in dataframe['Label'] for label in label_list}
    # Convert the set to a list to get your final list of unique labels
    return list(unique_labels)


Take the test, train, and validation directories for the `.conll` files, and turn them into their own `.pkl`

In [ ]:
# directories for the input .conll files
test_data_dir = "../../datasets/conll_2003/baseline/test"
train_data_dir = "../../datasets/conll_2003/baseline/train"
val_data_dir = "../../datasets/conll_2003/baseline/val"

# reuse the code from `tac_preprocessing.ipynb` to get an appropriately structured DataFrame
def format_df_from_dir(data_dir):
    data_df = dir_to_df(data_dir)
    data_df = add_sentence_numbers(data_df)
    data_grouped = data_df.groupby("Sentence #").agg({'Label': list, 'Start': list, 'End': list, 'Word': list})
    data_grouped = data_grouped.reset_index()
    return data_grouped[["Word", "Label"]]

test_df = format_df_from_dir(test_data_dir)
train_df = format_df_from_dir(train_data_dir)
val_df = format_df_from_dir(val_data_dir)

# pickle them
test_pkl_fpath = "../../datasets/conll_2003/baseline/test/test.pkl"
train_pkl_fpath = "../../datasets/conll_2003/baseline/train/train.pkl"
val_pkl_fpath = "../../datasets/conll_2003/baseline/val/val.pkl"

test_df.to_pickle(test_pkl_fpath)
train_df.to_pickle(train_pkl_fpath)
val_df.to_pickle(val_pkl_fpath)

Test out the pickles by reading them

In [ ]:
df = pd.read_pickle(train_pkl_fpath)
for row in df.iterrows():
    print(row)
